# Interactive Calibration Sessions

This is the canonical human workflow for the color-cue calibration paradigm. It supports both halves of the protocol in one place:

1. **Experiment 1:** estimate a subject's baseline perceptual threshold with saved staircase sessions.
2. Convert that threshold into per-stimulus internal noise, `sigma_internal`.
3. **Experiment 2:** choose target total per-stimulus noise levels, compute subject-specific `sigma_ext`, and run saved fixed-level 2AFC sessions.
4. Load saved sessions later and fit Gaussian 2AFC curves by `sigma_target`.

Use `noise-calibration-demo.ipynb` for a simulation-only conceptual demo. Use this notebook for human data collection.

In [ ]:
import numpy as np
import pandas as pd

from color_cue.interactive import (
    InteractiveColorCueTask,
    InteractiveTaskConfig,
    StaircaseColorCueTask,
    StaircaseTaskConfig,
    list_saved_sessions,
    list_saved_staircase_sessions,
    load_multiple_sessions,
    load_multiple_staircase_sessions,
    save_staircase_session,
    save_task_session,
    summarize_loaded_staircase_sessions,
)
from color_cue.psychophysics import (
    compute_external_noise,
    fit_all_gaussian_psychometrics,
    make_noise_calibration_table,
    plot_matched_noise_psychometrics,
    stimulus_sigma_from_jnd,
)

# Use a GUI backend before running tasks in Jupyter, for example:
# %matplotlib qt


## Shared Configuration

Set these identifiers once per subject. The two data roots keep baseline staircase sessions separate from calibrated experiment-2 sessions.

`STAIRCASE_P_CORRECT = 0.707` matches the approximate target performance of a 2-down 1-up staircase. That value is used when converting the final reversal threshold into `sigma_internal`.

In [ ]:
SUBJECT_ID = "subject_demo"

BASELINE_DATA_ROOT = "./session_data/baseline_staircase"
CALIBRATED_DATA_ROOT = "./session_data/calibrated_task"

THETA0 = -np.pi / 4
KAPPA = 0.1
STIMULUS_SIZE = (112, 112)
STAIRCASE_P_CORRECT = 0.707


## Experiment 1: Run A Baseline Staircase Session

The staircase is useful because it concentrates trials near the subject's threshold. A fixed grid spends many trials on values that are obviously easy or too hard; the staircase adapts `|delta_theta|` and gives a more efficient baseline threshold estimate.

For the baseline calibration, keep `sigma_ext = 0`. You can run separate sessions for `redder` and `bluer`; later cells can pool them or show them separately.

In [ ]:
BASELINE_SESSION_ID = "baseline_redder_001"
BASELINE_NOTES = "baseline 2-down 1-up staircase, sigma_ext=0"
BASELINE_CONTEXT = "redder"

baseline_cfg = StaircaseTaskConfig(
    theta0=THETA0,
    sigma_ext_levels=(0.0,),
    delta_grid=(0.02, 0.03, 0.045, 0.065, 0.09, 0.13, 0.18, 0.25),
    start_index=5,
    context=BASELINE_CONTEXT,
    shared_noise=False,
    max_trials_per_staircase=60,
    max_reversals_per_staircase=10,
    threshold_reversal_count=6,
    size=STIMULUS_SIZE,
    kappa=KAPPA,
    trial_seed=123,
    stimulus_seed=456,
)
baseline_task = StaircaseColorCueTask(baseline_cfg)

# Uncomment to run interactively.
# baseline_results = baseline_task.run()

# Uncomment after running to save.
# save_staircase_session(
#     baseline_task,
#     BASELINE_DATA_ROOT,
#     SUBJECT_ID,
#     BASELINE_SESSION_ID,
#     notes=BASELINE_NOTES,
#     overwrite=False,
# )


## Experiment 1: Load Baseline Sessions And Estimate `sigma_internal`

Load any subset of saved baseline sessions. If you leave `BASELINE_SESSION_IDS = None`, all saved baseline staircase sessions for this subject are loaded.

The notebook computes a reversal-based threshold for each staircase, converts it to per-stimulus internal noise, and uses the mean of valid estimates as the default `SIGMA_INTERNAL` for experiment 2.

In [ ]:
list_saved_staircase_sessions(BASELINE_DATA_ROOT, SUBJECT_ID)


In [ ]:
BASELINE_SESSION_IDS = None  # or use ["baseline_redder_001", "baseline_bluer_001"]

try:
    baseline_loaded = load_multiple_staircase_sessions(
        BASELINE_DATA_ROOT,
        SUBJECT_ID,
        BASELINE_SESSION_IDS,
    )
except ValueError as exc:
    baseline_loaded = pd.DataFrame()
    print(exc)

baseline_loaded.head()


In [ ]:
if not baseline_loaded.empty:
    baseline_summary = summarize_loaded_staircase_sessions(
        baseline_loaded,
        threshold_reversal_count=6,
    )
    baseline_summary["sigma_internal_estimate"] = stimulus_sigma_from_jnd(
        baseline_summary["threshold_estimate"],
        p_correct=STAIRCASE_P_CORRECT,
    )
    valid_baseline = baseline_summary.dropna(subset=["sigma_internal_estimate"])
    SIGMA_INTERNAL = float(valid_baseline["sigma_internal_estimate"].mean())
    display(baseline_summary)
    print(f"Using SIGMA_INTERNAL = {SIGMA_INTERNAL:.4f}")
else:
    baseline_summary = pd.DataFrame()
    SIGMA_INTERNAL = np.nan
    print("No baseline sessions loaded yet. Run and save experiment 1 before experiment 2.")


## Experiment 2: Compute Calibrated External Noise

Choose experimenter-defined target total per-stimulus noise levels. The notebook computes the subject-specific external noise values:

` sigma_ext = sqrt(sigma_target^2 - sigma_internal^2) `

Targets below the subject's baseline internal noise are infeasible and excluded from the experiment-2 task.

In [ ]:
SIGMA_TARGETS = (0.10, 0.15, 0.2)

if np.isfinite(SIGMA_INTERNAL):
    subject_noise = pd.DataFrame(
        {"subject_id": [SUBJECT_ID], "sigma_internal": [SIGMA_INTERNAL]}
    )
    calibration_table = make_noise_calibration_table(subject_noise, SIGMA_TARGETS)
    feasible_calibration = calibration_table[calibration_table["is_feasible"]].copy()
    display(calibration_table)
else:
    subject_noise = pd.DataFrame()
    calibration_table = pd.DataFrame()
    feasible_calibration = pd.DataFrame()
    print("Estimate SIGMA_INTERNAL first.")


## Experiment 2: Run And Save A Calibrated Fixed-Level Session

This task uses the feasible `sigma_ext` values from the calibration table. Before saving, results are annotated with `sigma_internal` and `sigma_target`, so future pooled analyses know exactly which total-noise condition each trial belongs to.

In [ ]:
CALIBRATED_SESSION_ID = "calibrated_001"
CALIBRATED_NOTES = "calibrated fixed-level 2AFC session"
DELTA_THETAS = (-0.20, -0.14, -0.08, -0.04, 0.04, 0.08, 0.14, 0.20)
CONTEXTS = ("redder",)
N_REPEATS = 6

if not feasible_calibration.empty:
    calibrated_cfg = InteractiveTaskConfig(
        theta0=THETA0,
        delta_thetas=DELTA_THETAS,
        sigma_ext_levels=tuple(feasible_calibration["sigma_ext"].astype(float)),
        n_repeats=N_REPEATS,
        contexts=CONTEXTS,
        size=STIMULUS_SIZE,
        kappa=KAPPA,
        trial_seed=789,
        stimulus_seed=987,
    )
    calibrated_task = InteractiveColorCueTask(calibrated_cfg)
else:
    calibrated_task = None
    print("No feasible calibrated noise levels available yet.")

# Uncomment to run interactively.
# calibrated_results = calibrated_task.run()

# Uncomment after running to annotate and save.
# annotated = calibrated_task.results.merge(
#     feasible_calibration[["sigma_internal", "sigma_target", "sigma_ext"]],
#     on="sigma_ext",
#     how="left",
# )
# calibrated_task._responses = annotated.to_dict(orient="records")
# save_task_session(
#     calibrated_task,
#     CALIBRATED_DATA_ROOT,
#     SUBJECT_ID,
#     CALIBRATED_SESSION_ID,
#     notes=CALIBRATED_NOTES,
#     overwrite=True,
# )


## Experiment 2: Load Calibrated Sessions

Load all saved calibrated sessions for this subject, or specify a subset. Pooling compatible sessions is the intended way to get stable Gaussian psychometric fits.

In [ ]:
list_saved_sessions(CALIBRATED_DATA_ROOT, SUBJECT_ID)


In [ ]:
CALIBRATED_SESSION_IDS = None  # or use ["calibrated_001", "calibrated_002"]

try:
    calibrated_loaded = load_multiple_sessions(
        CALIBRATED_DATA_ROOT,
        SUBJECT_ID,
        CALIBRATED_SESSION_IDS,
    )
except ValueError as exc:
    calibrated_loaded = pd.DataFrame()
    print(exc)

calibrated_loaded.head()


## Pooled Matched-Noise Analysis

Fit Gaussian 2AFC curves by subject, context, and `sigma_target`. If calibration is working, different subjects should recover similar `sigma_stimulus` values within the same target condition.

In [ ]:
if not calibrated_loaded.empty:
    calibrated_fits = fit_all_gaussian_psychometrics(
        calibrated_loaded,
        group_cols=("subject_id", "context", "sigma_target"),
    )
    display(
        calibrated_fits[
            [
                "subject_id",
                "context",
                "sigma_target",
                "sigma_stimulus",
                "threshold_75",
                "success",
            ]
        ]
    )
else:
    calibrated_fits = pd.DataFrame()
    print("No calibrated sessions loaded yet.")


In [ ]:
if not calibrated_loaded.empty:
    plot_matched_noise_psychometrics(
        calibrated_loaded,
        calibrated_fits,
        context="redder",
    );
